# Automotive Fuel Spending Insights

Charts built from the ONS weekly automotive fuel insights workbook.

## Setup and data

The chart functions remain in `fig_fuel.py` so the notebook and script share one implementation.

In [64]:
from pathlib import Path

import altair as alt
import pandas as pd

import eco_style
from eco_style import pallete

WORKSPACE = Path.cwd()
if WORKSPACE.name != "charts-week7sept":
    WORKSPACE = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == "charts-week7sept")

DATA_PATH = WORKSPACE / "data" / "automotivefuelinsightsdataset040926.xlsx"
WIDTH, HEIGHT = 600, 400
SRC = "Source: ONS real-time indicators, ONS calculations and Department for Energy Security and Net Zero, published 4 September 2026."

alt.data_transformers.disable_max_rows()
alt.theme.enable("report")


def read_ons(path, sheet, header_row=4):
    data = pd.read_excel(path, sheet_name=sheet, header=header_row).dropna(how="all").reset_index(drop=True)
    first = data.columns[0]
    data[first] = pd.to_datetime(data[first], errors="coerce")
    data = data[data[first].notna()].reset_index(drop=True)
    for column in data.columns[1:]:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    return data.rename(columns={first: "date"})


def weekly():
    data = read_ons(DATA_PATH, "1.Weekly Fuel Insights")
    data.columns = ["date", "price", "qty", "sales"]
    return data


def y_axis(field, title, values, domain):
    return alt.Y(field, title=title, scale=alt.Scale(domain=domain, nice=False), axis=alt.Axis(values=values, domain=True, grid=True, tickSize=0))


def x_axis(field, title=None, values=None, domain=None):
    return alt.X(field, title=title, scale=alt.Scale(domain=domain, nice=False) if domain else alt.Undefined, axis=alt.Axis(values=values, domain=True, grid=False, tickSize=0, titleAnchor="start", titleAlign="left", titleAngle=0))


def time_axis(values):
    return alt.X("date:T", title=None, axis=alt.Axis(format="%Y", tickCount="year", values=values, domain=True, grid=False, tickSize=0))


def horizontal_rule(value, encoding, dash=None, opacity=0.5):
    field = encoding.shorthand.split(":")[0]
    mark = {"color": pallete["domain"], "opacity": opacity}
    if dash:
        mark["strokeDash"] = list(dash)
    return alt.Chart(pd.DataFrame({field: [value]})).mark_rule(**mark).encode(y=encoding)


def labels(data, x, y, field, scale=None, color=None, dx=7, size=11, weight=600):
    chart = alt.Chart(data).mark_text(align="left", baseline="middle", dx=dx, fontSize=size, fontWeight=weight)
    return chart.encode(x=x, y=y, text=alt.Text(field), color=alt.Color(field, scale=scale, legend=None) if scale else alt.value(color or pallete["domain"]))


def notes(data, x, y):
    return alt.Chart(data).mark_text(align="left", baseline="middle", fontSize=10.5, fontWeight=400, color=pallete["Deemphasize_Other"]).encode(x=x, y=y, text="t:N")


def titled(plot, title, subtitle):
    return plot.properties(width=WIDTH, height=HEIGHT, title=alt.TitleParams(text=title, subtitle=subtitle, anchor="start", color=pallete["domain"], font="Circular Std", fontSize=16, subtitleColor=pallete["Deemphasize_Discrete"], subtitleFont="Circular Std", subtitleFontSize=12, subtitlePadding=8, offset=16)).configure_view(stroke="transparent")


def f1():
    data = weekly()[["date", "price", "qty"]].melt("date", var_name="kind", value_name="value")
    names = {"price": "Pump price", "qty": "Fuel per transaction"}
    data["series"] = data.kind.map(names)
    scale = alt.Scale(domain=list(names.values()), range=[pallete["nominal_2"], pallete["nominal_1"]])
    x = time_axis([f"{year}-01-01" for year in range(2021, 2027)])
    y = y_axis("value:Q", "Index, 100 = same week a year earlier", [80, 100, 120, 140], [72, 150])
    lines = alt.Chart(data).mark_line(strokeWidth=2).encode(x=x, y=y, color=alt.Color("series:N", scale=scale, legend=None))
    annotations = pd.DataFrame({"date": pd.to_datetime(["2022-08-20", "2023-11-01", "2025-09-01"]), "value": [143, 82, 133], "t": ["2022 oil shock", "Prices fell back through 2023", "Prices climbing again"]})
    return titled(horizontal_rule(100, y, dash=(4, 3)) + lines + labels(data[data.date == data.date.max()], x, y, "series:N", scale=scale) + notes(annotations, x, y), "As pump prices climb, drivers are putting less in the tank", ["UK average pump price and estimated fuel demanded per transaction, weekly", "January 2021 to 23 August 2026"])


def f2():
    data = weekly().dropna(subset=["price", "qty"]).copy()
    data["year"] = data.date.dt.year.astype(str)
    data["group"] = data.year.where(data.year.isin(["2022", "2026"]), "2021, 2023–2025")
    order = ["2021, 2023–2025", "2022", "2026"]
    scale = alt.Scale(domain=order, range=[pallete["Other_3"], pallete["nominal_5"], pallete["nominal_2"]])
    x = x_axis("price:Q", "Pump price index", [80, 100, 120, 140], [70, 152])
    y = y_axis("qty:Q", "Fuel per transaction, index", [80, 90, 100, 110], [78, 116])
    points = alt.Chart(data).mark_point(filled=True, size=38, opacity=0.85).encode(x=x, y=y, color=alt.Color("group:N", scale=scale, legend=None, sort=order))
    annotations = pd.DataFrame({"price": [137, 78, 127], "qty": [104, 114, 87], "t": ["2022", "2021 and 2023–2025", "2026"]})
    annotation_scale = alt.Scale(domain=["2022", "2021 and 2023–2025", "2026"], range=[pallete["nominal_5"], pallete["Deemphasize_Other"], pallete["nominal_2"]])
    plot = horizontal_rule(100, y, dash=(3, 3), opacity=0.35) + points + labels(annotations, x, y, "t:N", scale=annotation_scale, size=11.5, weight=700) + notes(pd.DataFrame({"price": [73], "qty": [82], "t": ["Each dot is one week"]}), x, y)
    return titled(plot, "Higher pump prices go with smaller fill-ups, week after week", ["Estimated fuel demanded per transaction against the average UK pump price", "Weekly, January 2021 to 23 August 2026"])


def f3():
    data = weekly().copy()
    iso = data.date.dt.isocalendar()
    data["year"] = iso.year.astype(str)
    data["week"] = iso.week.astype(int)
    data = data[(data.week <= 52) & (data.year != "2020")]
    order = ["2021", "2022", "2023", "2024", "2025", "2026"]
    scale = alt.Scale(domain=order, range=[pallete["Other_3"], pallete["nominal_5"], pallete["Other_3"], pallete["Other_3"], pallete["Other_3"], pallete["nominal_2"]])
    x = x_axis("week:Q", "Week of year", [1, 10, 20, 30, 40, 52], [1, 52])
    y = y_axis("price:Q", "Index, 100 = same week a year earlier", [80, 100, 120, 140], [72, 150])
    lines = alt.Chart(data).mark_line(strokeWidth=2).encode(x=x, y=y, color=alt.Color("year:N", scale=scale, legend=None, sort=order), detail="year:N")
    tips = data.sort_values("week").groupby("year").tail(1)
    plot = horizontal_rule(100, y, dash=(4, 3)) + lines + labels(tips[tips.year.isin(["2022", "2026"])], x, y, "year:N", scale=scale) + labels(tips[~tips.year.isin(["2022", "2026"])], x, y, "year:N", color=pallete["Deemphasize_Other"], dx=9, size=10, weight=400)
    return titled(plot, "Pump price inflation is back above 20% for the first time since 2022", ["UK average pump price by week of year, weekly", "Grey lines: 2021, 2023, 2024 and 2025"])


def f4():
    data = weekly()[["date", "qty"]].dropna().copy()
    data["above"] = data.qty.clip(lower=100)
    data["below"] = data.qty.clip(upper=100)
    x = time_axis([f"{year}-01-01" for year in range(2021, 2027)])
    y = y_axis("above:Q", "Index, 100 = same week a year earlier", [80, 90, 100, 110], [78, 118])
    y_below = y_axis("below:Q", "Index, 100 = same week a year earlier", [80, 90, 100, 110], [78, 118])
    above = alt.Chart(data).mark_area(color=pallete["nominal_1"], opacity=0.9).encode(x=x, y=y, y2=alt.datum(100))
    below = alt.Chart(data).mark_area(color=pallete["nominal_2"], opacity=0.9).encode(x=x, y=y_below, y2=alt.datum(100))
    annotations = pd.DataFrame({"date": pd.to_datetime(["2024-05-01", "2021-05-01", "2025-11-01"]), "above": [114, 84, 86], "t": ["Bigger fill-ups than a year earlier", "Smaller fill-ups", "21 straight weeks below"]})
    return titled(above + below + horizontal_rule(100, y) + notes(annotations, x, y), "Fuel bought per fill-up has been below year-earlier levels since April", ["Estimated fuel demanded per transaction, weekly", "Blue: above the level of a year earlier. Red: below"])


data = weekly()
data.head()

,date,price,qty,sales
0,2021-01-31,92.609795,97.202065,65.756879
1,2021-02-07,94.467352,95.448788,67.471288
2,2021-02-14,95.851356,95.891945,67.300106
3,2021-02-21,97.410857,92.916855,69.005771
4,2021-02-28,98.312732,92.083632,69.591091


In [65]:
data.shape, data.columns.tolist(), data[["date", "price", "qty"]].describe()

((291, 4),
 ['date', 'price', 'qty', 'sales'],
                       date       price         qty
 count                  291  291.000000  291.000000
 mean   2023-11-12 00:00:00  105.783190   98.419033
 min    2021-01-31 00:00:00   74.218966   79.834487
 25%    2022-06-22 12:00:00   92.853634   94.068582
 50%    2023-11-12 00:00:00  101.037329   99.030087
 75%    2025-04-02 12:00:00  120.160957  102.680194
 max    2026-08-23 00:00:00  146.764208  114.021890
 std                    NaN   16.206941    5.824175)

## Preview and save charts

Run this cell to display each Altair chart inline before exporting PNG and SVG files.

In [66]:
from IPython.display import display

chart_1 = f1()
display(chart_1)
output_dir = WORKSPACE / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
png_path = output_dir / "fuel_1_price_vs_demand.png"
svg_path = output_dir / "fuel_1_price_vs_demand.svg"
chart_1.save(str(png_path), scale_factor=3)
chart_1.save(str(svg_path))
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")

alt.LayerChart(...)

Saved: /Users/alonso/Documents/GitHub/RADataHub/ChartOfTheDay/charts-week7sept/outputs/fuel_1_price_vs_demand.png
Saved: /Users/alonso/Documents/GitHub/RADataHub/ChartOfTheDay/charts-week7sept/outputs/fuel_1_price_vs_demand.svg


In [60]:
chart_2 = f2()
display(chart_2)
png_path = output_dir / "fuel_2_scatter.png"
svg_path = output_dir / "fuel_2_scatter.svg"
chart_2.save(str(png_path), scale_factor=3)
chart_2.save(str(svg_path))
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")

alt.LayerChart(...)

Saved: /Users/alonso/Documents/GitHub/RADataHub/ChartOfTheDay/charts-week7sept/outputs/fuel_2_scatter.png
Saved: /Users/alonso/Documents/GitHub/RADataHub/ChartOfTheDay/charts-week7sept/outputs/fuel_2_scatter.svg


In [56]:
chart_3 = f3()
display(chart_3)
png_path = output_dir / "fuel_3_cycle.png"
svg_path = output_dir / "fuel_3_cycle.svg"
chart_3.save(str(png_path), scale_factor=3)
chart_3.save(str(svg_path))
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")

alt.LayerChart(...)

Saved: /Users/alonso/Documents/GitHub/RADataHub/ChartOfTheDay/charts-week7sept/outputs/fuel_3_cycle.png
Saved: /Users/alonso/Documents/GitHub/RADataHub/ChartOfTheDay/charts-week7sept/outputs/fuel_3_cycle.svg


In [46]:
chart_4 = f4()
display(chart_4)
png_path = output_dir / "fuel_4_deviation.png"
svg_path = output_dir / "fuel_4_deviation.svg"
chart_4.save(str(png_path), scale_factor=3)
chart_4.save(str(svg_path))
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")

alt.LayerChart(...)

Saved: /Users/alonso/Documents/GitHub/RADataHub/ChartOfTheDay/charts-week7sept/outputs/fuel_4_deviation.png
Saved: /Users/alonso/Documents/GitHub/RADataHub/ChartOfTheDay/charts-week7sept/outputs/fuel_4_deviation.svg
